# Hinglish Sentiment Analysis

This notebook implements transformer-based sentiment analysis on Hinglish tweets, including model fine-tuning and ensemble optimization.

## 1 · Install Dependencies

In [ ]:
!pip install -q -U transformers datasets scikit-learn scipy huggingface_hub

## 2 · Imports & HF Login

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import minimize
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import login

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device      :', DEVICE)
print('GPU count   :', torch.cuda.device_count())



## 3 · Load & Clean Data

In [ ]:
TRAIN_PATH = '/kaggle/input/datasets/vijayeswaran/hinglish-iai-datasets/FinalTrainingOnly.tsv'
VAL_PATH   = '/kaggle/input/datasets/vijayeswaran/hinglish-iai-datasets/ValidationOnly.tsv'

train = pd.read_csv(TRAIN_PATH, sep='\t', header=None, names=['uid', 'text', 'label'])
val   = pd.read_csv(VAL_PATH,   sep='\t', header=None, names=['uid', 'text', 'label'])

def clean_text(text):
    text = str(text)
    text = re.sub(r'@\w+', '', text)
    text = text.replace('nan', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['text'] = train['text'].apply(clean_text)
val['text']   = val['text'].apply(clean_text)

train = train.dropna(subset=['text', 'label'])
val   = val.dropna(subset=['text', 'label'])

train = train[train['text'].str.strip() != ''].reset_index(drop=True)
val   = val[val['text'].str.strip()   != ''].reset_index(drop=True)

train['label'] = train['label'].astype(int)
val['label']   = val['label'].astype(int)

print(f'Train samples : {len(train)}')
print(f'Val samples   : {len(val)}')
print(f'Classes       : {sorted(train["label"].unique())}')
print(f'\nTrain label distribution:\n{train["label"].value_counts().sort_index()}')
print(f'\nVal label distribution:\n{val["label"].value_counts().sort_index()}')

Train samples : 14569
Val samples   : 2999
Classes       : [np.int64(0), np.int64(1), np.int64(2)]

Train label distribution:
label
0    4250
1    5454
2    4865
Name: count, dtype: int64

Val label distribution:
label
0     890
1    1127
2     982
Name: count, dtype: int64


## 4 · Config

In [ ]:
MODELS = {
    'mbert'      : 'bert-base-multilingual-cased',
    'xlm_roberta': 'xlm-roberta-base',
    'muril'      : 'google/muril-base-cased',
}

NUM_LABELS = 3
MAX_LEN    = 128
BATCH      = 16
EPOCHS     = 8
LR         = 3e-5

# Optimisation objective: ALPHA * F1 + (1-ALPHA) * Accuracy
# 0.6 slightly favours F1 to handle class imbalance better
ALPHA = 0.6

print('Models to train:')
for k, v in MODELS.items():
    print(f'  {k:<15} -> {v}')

Models to train:
  mbert           -> bert-base-multilingual-cased
  xlm_roberta     -> xlm-roberta-base
  muril           -> google/muril-base-cased


## 5 · Helper: Build HF Dataset (tokenize first, then wrap)

In [ ]:
def make_hf_dataset(texts, labels, tokenizer):
    """
    Tokenise a list of texts and return a HuggingFace Dataset
    ready for the Trainer. Uses from_dict to avoid the
    from_pandas + map column-persistence bug in newer datasets versions.
    """
    enc  = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
    )
    data = {k: v for k, v in enc.items()}
    data['label'] = list(labels)
    ds   = HFDataset.from_dict(data)

    fmt_cols = ['input_ids', 'attention_mask', 'label']
    if 'token_type_ids' in ds.column_names:
        fmt_cols.append('token_type_ids')
    ds.set_format(type='torch', columns=fmt_cols)
    return ds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

## 6 · Train All 3 Models

In [ ]:
def train_and_save(name, model_path):
    print(f'\n{"-"*60}')
    print(f'  Model    : {name}')
    print(f'  Hub path : {model_path}')
    print(f'{"-"*60}')

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        token=HF_TOKEN if HF_TOKEN else None,
        trust_remote_code=True,
    )

    train_ds = make_hf_dataset(
        train['text'].tolist(),
        train['label'].tolist(),
        tokenizer,
    )
    val_ds = make_hf_dataset(
        val['text'].tolist(),
        val['label'].tolist(),
        tokenizer,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
        token=HF_TOKEN if HF_TOKEN else None,
        trust_remote_code=True,
    )

    args = TrainingArguments(
        output_dir=f'./ckpt_{name}',
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=200,
        report_to='none',
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=4,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    save_dir = f'./{name}_finetuned'
    trainer.save_model(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f'  Saved to: {save_dir} ✅')

    del model
    torch.cuda.empty_cache()
    return save_dir


# --- Run training ---
saved = {}
for name, path in MODELS.items():
    saved[name] = train_and_save(name, path)

print('\n✅ All 3 models trained and saved.')
for k, v in saved.items():
    print(f'  {k:<15} -> {v}')

## 7 · Extract Softmax Probabilities on Validation Set

In [ ]:
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors='pt',
        )
    def __len__(self):
        return self.enc['input_ids'].shape[0]
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.enc.items()}


@torch.no_grad()
def get_softmax_probs(save_dir, texts, batch_size=32):
    tokenizer = AutoTokenizer.from_pretrained(save_dir)
    model     = AutoModelForSequenceClassification.from_pretrained(save_dir).to(DEVICE)
    model.eval()

    ds     = InferenceDataset(texts, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size)
    probs  = []

    for batch in loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs.append(torch.softmax(logits, dim=-1).cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.vstack(probs)


# --- Extract probs for every model ---
val_texts   = val['text'].tolist()
true_labels = val['label'].values

val_probs = {}
for name, save_dir in saved.items():
    print(f'Extracting probs: {name} ...')
    val_probs[name] = get_softmax_probs(save_dir, val_texts)
    p   = val_probs[name].argmax(axis=1)
    acc = accuracy_score(true_labels, p)
    f1  = f1_score(true_labels, p, average='weighted')
    print(f'  Individual -> Accuracy: {acc:.4f}  F1: {f1:.4f}')

model_names = list(val_probs.keys())
val_stack   = np.stack([val_probs[n] for n in model_names], axis=2)  # (N, 3, 3)
print(f'\nval_stack shape: {val_stack.shape}  ✅')

## 8 · Optimised Weight Search (Scipy)

Finds the 3 weights `w1, w2, w3` (must sum to 1, all ≥ 0) that maximise:
```
0.6 × F1_weighted  +  0.4 × Accuracy
```
Runs from 7 different starting points so we don't get stuck in a local minimum.

In [ ]:
def neg_combined_score(weights):
    w = np.clip(weights, 0, None)
    if w.sum() == 0:
        return 0.0
    w        = w / w.sum()
    combined = (val_stack * w).sum(axis=2)   # (N, 3)
    preds    = combined.argmax(axis=1)
    f1       = f1_score(true_labels,  preds, average='weighted')
    acc      = accuracy_score(true_labels, preds)
    return -(ALPHA * f1 + (1 - ALPHA) * acc)


starts = [
    [1/3, 1/3, 1/3],
    [0.5, 0.3, 0.2],
    [0.2, 0.5, 0.3],
    [0.3, 0.2, 0.5],
    [0.6, 0.2, 0.2],
    [0.2, 0.6, 0.2],
    [0.2, 0.2, 0.6],
]

constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
bounds      = [(0.0, 1.0)] * 3
best_res    = None

for w0 in starts:
    res = minimize(
        neg_combined_score,
        np.array(w0, dtype=float),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'ftol': 1e-10, 'maxiter': 1000},
    )
    if best_res is None or res.fun < best_res.fun:
        best_res = res

optimal_w  = np.clip(best_res.x, 0, None)
optimal_w /= optimal_w.sum()

print('Optimal Weights (Scipy):')
for n, w in zip(model_names, optimal_w):
    print(f'  {n:<15}  {w:.4f}')

Optimal Weights (Scipy):
  mbert            0.5000
  xlm_roberta      0.3000
  muril            0.2000


## 9 · Final Results

In [ ]:
def evaluate(weights, label):
    w = np.clip(np.array(weights, dtype=float), 0, None)
    w /= w.sum()
    preds = (val_stack * w).sum(axis=2).argmax(axis=1)
    acc   = accuracy_score(true_labels, preds)
    f1    = f1_score(true_labels, preds, average='weighted')
    print(f'  {label:<30}  Accuracy: {acc:.4f}   F1: {f1:.4f}')
    return acc, f1


print('=' * 65)
print('FINAL RESULTS ON VALIDATION SET')
print('=' * 65)

print('\nIndividual models:')
for name in model_names:
    p   = val_probs[name].argmax(axis=1)
    acc = accuracy_score(true_labels, p)
    f1  = f1_score(true_labels, p, average='weighted')
    print(f'  {name:<30}  Accuracy: {acc:.4f}   F1: {f1:.4f}')

print('\nEnsemble comparisons:')
evaluate([1/3, 1/3, 1/3], 'Equal weights (baseline)')
best_acc, best_f1 = evaluate(optimal_w, 'Optimised weights')

print('\nFinal Optimal Weights:')
for n, w in zip(model_names, optimal_w):
    print(f'  {n:<15}  {w:.4f}')

print(f'\nBest Ensemble -> Accuracy: {best_acc:.4f}   F1: {best_f1:.4f}')

FINAL RESULTS ON VALIDATION SET

Individual models:
  mbert                           Accuracy: 0.9020   F1: 0.9018
  xlm_roberta                     Accuracy: 0.8459   F1: 0.8451
  muril                           Accuracy: 0.8710   F1: 0.8707

Ensemble comparisons:
  Equal weights (baseline)        Accuracy: 0.9000   F1: 0.8997
  Optimised weights               Accuracy: 0.9090   F1: 0.9088

Final Optimal Weights:
  mbert            0.5000
  xlm_roberta      0.3000
  muril            0.2000

Best Ensemble -> Accuracy: 0.9090   F1: 0.9088
